# GPT-2：因果语言模型

这个 Notebook 从零实现 GPT-2 架构并在 TinyShakespeare 上训练字符级语言模型。

内容包括：
- 可学习绝对位置 Embedding
- Causal Multi-head Self-Attention（因果掩码）
- GeLU 激活的 MLP
- Post-Norm vs Pre-Norm 对比
- 训练与 Perplexity 监控
- 文本生成
- GPT-2 与 LLaMA 的架构差异对比

## 1. 环境准备

```bash
pip install torch
```

In [ ]:
import math
import requests
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 数据
    data_url: str   = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    seq_len: int    = 256
    batch_size: int = 64
    # 模型（GPT-2 small 风格，缩小版）
    vocab_size: int  = 65   # 字符级词表
    n_layers: int    = 6
    d_model: int     = 256
    n_heads: int     = 8
    d_ff: int        = 1024  # GPT-2 的 d_ff = 4 × d_model
    dropout: float   = 0.1
    # 训练
    lr: float   = 3e-4
    epochs: int = 5


cfg = Config()
cfg

## 2. 数据集：TinyShakespeare（字符级）

In [ ]:
try:
    text = requests.get(cfg.data_url, timeout=10).text
    print(f'Downloaded: {len(text):,} characters')
except Exception:
    text = 'To be or not to be, that is the question. ' * 2000
    print('Using fallback text.')

chars   = sorted(set(text))
char2id = {c: i for i, c in enumerate(chars)}
id2char = {i: c for c, i in char2id.items()}
cfg.vocab_size = len(chars)

encode = lambda s: [char2id[c] for c in s]
decode = lambda ids: ''.join(id2char[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n    = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f'Vocab size: {cfg.vocab_size}  |  Train tokens: {len(train_data):,}  |  Val tokens: {len(val_data):,}')

In [ ]:
class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data    = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        chunk = self.data[idx: idx + self.seq_len + 1]
        return chunk[:-1], chunk[1:]


train_loader = DataLoader(CharDataset(train_data, cfg.seq_len), batch_size=cfg.batch_size, shuffle=True)
val_loader   = DataLoader(CharDataset(val_data,   cfg.seq_len), batch_size=cfg.batch_size, shuffle=False)

x, y = next(iter(train_loader))
print('input:', x.shape, '  target:', y.shape)

## 3. GPT-2 架构实现

GPT-2 是纯 Decoder 架构（无 Encoder），核心组件：
- **Token Embedding** + **可学习绝对位置 Embedding**
- **Causal Self-Attention**：上三角掩码防止看到未来 token
- **MLP**：`Linear → GeLU → Linear`，中间维度为 `4 × d_model`
- **LayerNorm**（Post-Norm；GPT-2 原版；LLaMA 改为 Pre-Norm + RMSNorm）

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.n_heads  = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        # Q/K/V 合并成一个线性层，效率更高
        self.qkv  = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.proj_drop = nn.Dropout(cfg.dropout)
        # 因果掩码：上三角全 -inf，注册为 buffer
        mask = torch.triu(torch.ones(cfg.seq_len, cfg.seq_len) * float('-inf'), diagonal=1)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        # 变形为多头形式
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # 缩放点积注意力 + 因果掩码
        scale = self.head_dim ** -0.5
        attn  = (q * scale) @ k.transpose(-2, -1)
        attn  = attn + self.mask[:T, :T]  # 广播到 (B, heads, T, T)
        attn  = F.softmax(attn, dim=-1)
        attn  = self.attn_drop(attn)

        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.proj_drop(self.proj(out))

In [ ]:
class MLP(nn.Module):
    """GPT-2 的 FFN：Linear → GeLU → Linear，中间维度 4×d_model。"""
    def __init__(self, cfg):
        super().__init__()
        self.fc1  = nn.Linear(cfg.d_model, cfg.d_ff)
        self.fc2  = nn.Linear(cfg.d_ff,   cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


class GPT2Block(nn.Module):
    """GPT-2 原版使用 Pre-LN（先归一化，再做 Attn/MLP）。"""
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.attn  = CausalSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.mlp   = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class GPT2Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Token embedding + 可学习绝对位置 embedding（与 LLaMA 的 RoPE 不同）
        self.tok_embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_embed = nn.Embedding(cfg.seq_len,    cfg.d_model)
        self.drop      = nn.Dropout(cfg.dropout)
        self.blocks    = nn.ModuleList([GPT2Block(cfg) for _ in range(cfg.n_layers)])
        self.norm      = nn.LayerNorm(cfg.d_model)
        self.head      = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        # 权重共享
        self.head.weight = self.tok_embed.weight

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx):
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device).unsqueeze(0)
        # token embedding + position embedding
        x = self.drop(self.tok_embed(idx) + self.pos_embed(pos))
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x)


model = GPT2Model(cfg).to(device)
model

## 4. 各层 Tensor 尺寸分析

In [ ]:
@torch.no_grad()
def inspect_shapes(model, cfg):
    idx = torch.randint(0, cfg.vocab_size, (2, cfg.seq_len))
    pos = torch.arange(cfg.seq_len).unsqueeze(0)

    tok = model.tok_embed(idx)
    pe  = model.pos_embed(pos)
    print(f'token embed    : {tuple(tok.shape)}')
    print(f'pos embed      : {tuple(pe.shape)}')
    x   = model.drop(tok + pe)
    print(f'input to blocks: {tuple(x.shape)}')

    blk = model.blocks[0]
    qkv = blk.attn.qkv(blk.norm1(x))
    q, k, v = qkv.split(cfg.d_model, dim=-1)
    B, T = x.shape[:2]
    q_h = q.view(B, T, cfg.n_heads, -1).transpose(1, 2)
    print(f'Q/K/V heads    : {tuple(q_h.shape)}  (B, n_heads, T, head_dim)')

    logits = model(idx)
    print(f'logits         : {tuple(logits.shape)}')


inspect_shapes(model.cpu(), cfg)
model = model.to(device)

## 5. GPT-2 vs LLaMA 架构对比

| 组件 | GPT-2 | LLaMA 2/3 |
|------|-------|----------|
| 位置编码 | 可学习绝对 PE | RoPE（旋转相对位置） |
| 归一化 | Pre-LayerNorm | Pre-RMSNorm |
| Attention | MHA（全头 KV） | GQA（少量 KV 头） |
| FFN 激活 | GeLU | SwiGLU（门控） |
| FFN 结构 | 2 个矩阵 | 3 个矩阵 |
| Bias | 有（大部分层） | 无 |
| 外推能力 | 弱（固定长度 PE） | 强（RoPE） |

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f'Trainable parameters: {count_parameters(model):,}')

## 6. 训练函数

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.1)


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total      = 0
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total      += x.size(0)
    return total_loss / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total      = 0
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        total_loss += loss.item() * x.size(0)
        total      += x.size(0)
    return total_loss / total

## 7. 训练主循环

In [ ]:
history = {'train_loss': [], 'val_loss': []}

for epoch in range(cfg.epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss   = evaluate(model, val_loader, device)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(
        f'Epoch [{epoch + 1}/{cfg.epochs}]  '
        f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  '
        f'train_ppl={math.exp(train_loss):.2f}  val_ppl={math.exp(val_loss):.2f}'
    )

In [ ]:
epochs_r = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_r, history['train_loss'], label='train')
axes[0].plot(epochs_r, history['val_loss'],   label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_r, [math.exp(l) for l in history['train_loss']], label='train')
axes[1].plot(epochs_r, [math.exp(l) for l in history['val_loss']],   label='val')
axes[1].set_title('Perplexity')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. 文本生成

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, temperature=1.0, top_k=50, device='cpu'):
    model.eval()
    idx = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        # GPT-2 位置 embedding 有长度限制，超出时截断
        idx_cond = idx[:, -cfg.seq_len:]
        logits   = model(idx_cond)[:, -1, :] / temperature

        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, -1:]] = float('-inf')

        probs    = F.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1)
        idx      = torch.cat([idx, next_tok], dim=1)

    return decode(idx[0].tolist())


prompts = ['HAMLET:', 'First Citizen:', 'To be']

for temp in [0.7, 1.0]:
    print(f'=== temperature={temp}, top_k=40 ===\n')
    for p in prompts:
        out = generate(model, p, max_new_tokens=120, temperature=temp, top_k=40, device=device)
        print(out)
        print('-' * 60)
    print()